In [1]:
import os
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

In [2]:
rolling_num=10

In [3]:
pd.set_option('display.max_columns',900)

In [4]:
df=pd.read_csv('df.csv')

C:\Users\enabi\anaconda3\envs\untitled\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (102) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [5]:
del df['Unnamed: 0']

In [6]:
df2=df.copy()

In [7]:
df2.shape

(7508, 165)

In [8]:
col= df2.loc[:,'Date':'Away_Bookings_Points'].columns
#col

In [9]:
df2= df2.loc[:,list(col)+['Final_result','Home_score','Away_score']]
df2.head(2)

,Date,HomeTeam,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score
0,2000-08-19,Charlton,Man City,4.0,0.0,2.0,0.0,1,20043.0,Rob Harris,17.0,8.0,14.0,4.0,2.0,1.0,6.0,6.0,13.0,12.0,8.0,6.0,1.0,2.0,0.0,0.0,10.0,20.0,1.0,3.0,0.0
1,2000-08-19,Chelsea,West Ham,4.0,2.0,1.0,0.0,1,34914.0,Graham Barber,17.0,12.0,10.0,5.0,1.0,0.0,7.0,7.0,19.0,14.0,2.0,3.0,1.0,2.0,0.0,0.0,10.0,20.0,1.0,3.0,0.0


In [10]:
df2.sort_values('Date', inplace=True)

In [11]:
shh = df2.groupby(['HomeTeam']).shift()
shh.head(5)

,Date,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
shh = df2.groupby(['HomeTeam']).shift()
del shh['Date']
sha = df2.groupby(['AwayTeam']).shift()
del sha['Date']
temp_H =pd.DataFrame(df2[['Date','HomeTeam']]).join(shh)
temp_A =pd.DataFrame(df2[['Date','AwayTeam']]).join(sha)

In [13]:
temp_H=temp_H.set_index('Date')
temp_A=temp_A.set_index('Date')

In [14]:
res1= pd.merge(df2, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(), suffixes=['','_HomeTeam_sum_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [15]:
res1= pd.merge(res1, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(), suffixes=['','_AwayTeam_sum_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [16]:
res1= pd.merge(res1, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).median().reset_index(), suffixes=['','_HomeTeam_median_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [17]:
res1= pd.merge(res1, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).median().reset_index(), suffixes=['','_AwayTeam_median_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [18]:
res1= pd.merge(res1, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(), suffixes=['','_HomeTeam_average_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [19]:
res1= pd.merge(res1, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(), suffixes=['','_AwayTeam_average_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [20]:
df3= df2[['Date','AwayTeam','HomeTeam','Final_Away_goals','Final_Home_goals','Half_Time_Away_Goals','Half_Time_Home_Goals','Half_Time_Result','Crowd_Attendance','Referee','Away_Shots','Home_Shots','Away_Shots_Target','Home_Shots_Target','Away_Woodwork','Home_Woodwork','Away_Corners','Home_Corners','Away_Fouls','Home_Fouls','Away_Offsides','Home_Offsides','Away_Yellow_Cards','Home_Yellow_Cards','Away_Red_Cards','Home_Red_Cards','Away_Bookings_Points','Home_Bookings_Points', 'Final_result','Away_score','Home_score']]

In [21]:
df4= df3.rename(columns = {'AwayTeam':'HomeTeam','HomeTeam':'AwayTeam', 'Final_Away_goals':'Final_Home_goals', 'Final_Home_goals':'Final_Away_goals', 'Half_Time_Home_Goals':'Half_Time_Away_Goals','Home_Shots':'Away_Shots','Home_Shots_Target':'Away_Shots_Target','Home_Woodwork':'Away_Woodwork','Home_Corners':'Away_Corners','Home_Fouls':'Away_Fouls','Home_Offsides':'Away_Offsides','Home_Yellow_Cards':'Away_Yellow_Cards','Home_Red_Cards':'Away_Red_Cards','Home_Bookings_Points':'Away_Bookings_Points', 'Half_Time_Away_Goals':'Half_Time_Home_Goals','Away_Shots':'Home_Shots','Away_Shots_Target':'Home_Shots_Target','Away_Woodwork':'Home_Woodwork','Away_Corners':'Home_Corners','Away_Fouls':'Home_Fouls','Away_Offsides':'Home_Offsides','Away_Yellow_Cards':'Home_Yellow_Cards','Away_Red_Cards':'Home_Red_Cards','Away_Bookings_Points':'Home_Bookings_Points', 'Away_score':'Home_score', 'Home_score':'Away_score'})

In [22]:
filt1= df4['Final_result']==1
filt3= df4['Final_result']==3
df4.loc[filt1,'Final_result']= 3
df4.loc[filt3,'Final_result']= 1
filt2= df4['Half_Time_Result']==1
filt4= df4['Half_Time_Result']==3
df4.loc[filt2,'Half_Time_Result']= 3
df4.loc[filt4,'Half_Time_Result']= 1

In [23]:
df_cc= pd.concat([df2, df4])

In [24]:
df_cc = df_cc.reset_index()
del df_cc['index']

In [25]:
shh = df_cc.groupby(['HomeTeam']).shift()
del shh['Date']
sha = df_cc.groupby(['AwayTeam']).shift()
del sha['Date']
temp_H =pd.DataFrame(df_cc[['Date','HomeTeam']]).join(shh)
temp_A =pd.DataFrame(df_cc[['Date','AwayTeam']]).join(sha)

In [26]:
temp_H=temp_H.set_index('Date')
temp_A=temp_A.set_index('Date')

In [27]:
cc1= pd.merge(df_cc, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(), suffixes=['','_Home_AllGames_sum_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [28]:
cc1= pd.merge(cc1, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).median().reset_index(), suffixes=['','_Home_AllGames_median_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [29]:
# df_cc_home= rolling (average) on the df_cc for the home column & merge with cc2
cc1= pd.merge(cc1, temp_H.groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(), suffixes=['','_Home_AllGames_Average_'+str(rolling_num)], on=['HomeTeam', 'Date'])

In [30]:
cols= set(cc1.columns.to_list()) - set(cc1.loc[:,'AwayTeam':'Away_score'].columns.to_list())

In [31]:
df_home=cc1.loc[:,cols]

In [32]:
# res7 = merge res6 and df_home based on the df_cc_home.home=res6.home and date
res7= pd.merge(res1, df_home, on=['HomeTeam', 'Date'])

In [33]:
# rolling (sum) on the df_cc for the away column & merge with df_cc
aa1= pd.merge(df_cc, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(), suffixes=['','_Away_AllGames_sum_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [34]:
# df_cc_home= rolling (median) on the df_cc for the away column & merge with aa1
aa1= pd.merge(aa1, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).median().reset_index(), suffixes=['','_Away_AllGames_median_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [35]:
# df_cc_home= rolling (average) on the df_cc for the away column & merge with aa2
aa1= pd.merge(aa1, temp_A.groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(), suffixes=['','_Away_AllGames_Average_'+str(rolling_num)], on=['AwayTeam', 'Date'])

In [36]:
col= aa1.loc[:,'Final_Home_goals':'Away_score'].columns
col

Index(['Final_Home_goals', 'Final_Away_goals', 'Half_Time_Home_Goals',
       'Half_Time_Away_Goals', 'Half_Time_Result', 'Crowd_Attendance',
       'Referee', 'Home_Shots', 'Away_Shots', 'Home_Shots_Target',
       'Away_Shots_Target', 'Home_Woodwork', 'Away_Woodwork', 'Home_Corners',
       'Away_Corners', 'Home_Fouls', 'Away_Fouls', 'Home_Offsides',
       'Away_Offsides', 'Home_Yellow_Cards', 'Away_Yellow_Cards',
       'Home_Red_Cards', 'Away_Red_Cards', 'Home_Bookings_Points',
       'Away_Bookings_Points', 'Final_result', 'Home_score', 'Away_score'],
      dtype='object')

In [37]:
list(col)+['HomeTeam']

['Final_Home_goals',
 'Final_Away_goals',
 'Half_Time_Home_Goals',
 'Half_Time_Away_Goals',
 'Half_Time_Result',
 'Crowd_Attendance',
 'Referee',
 'Home_Shots',
 'Away_Shots',
 'Home_Shots_Target',
 'Away_Shots_Target',
 'Home_Woodwork',
 'Away_Woodwork',
 'Home_Corners',
 'Away_Corners',
 'Home_Fouls',
 'Away_Fouls',
 'Home_Offsides',
 'Away_Offsides',
 'Home_Yellow_Cards',
 'Away_Yellow_Cards',
 'Home_Red_Cards',
 'Away_Red_Cards',
 'Home_Bookings_Points',
 'Away_Bookings_Points',
 'Final_result',
 'Home_score',
 'Away_score',
 'HomeTeam']

In [38]:
cols= set(aa1.columns.to_list()) - set(aa1.loc[:,list(col)+['HomeTeam']].columns.to_list())

In [39]:
df_away=aa1.loc[:,cols]

In [40]:
# merge res7 and df_away based on the df_away.away=res7.away and date
df_final= pd.merge(res7, df_away, on=['AwayTeam','Date'])

In [41]:
df_final.to_csv('final_2.csv')

In [42]:
# Rolling (sum) df2 on 'AwayTeam' and merge with df2 only on "awayteam" without date
rex1= pd.merge(df2, df2.set_index('Date').groupby(['AwayTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Away_of_home_sum'], left_on=['HomeTeam'],right_on=['AwayTeam'])

In [43]:
rex1['menha_date']= (pd.to_datetime(rex1['Date'])-pd.to_datetime(rex1['Date_'+str(rolling_num)+'Away_of_home_sum'])).dt.days

In [44]:
filt= rex1['menha_date']>0
rex2 = rex1.loc[filt]

In [45]:
rex2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index()

,Date,HomeTeam,menha_date
0,2000-08-21,Arsenal,2
1,2000-08-22,Bradford,3
2,2000-08-22,Ipswich,3
3,2000-08-22,Middlesbrough,3
4,2000-08-23,Everton,4
...,...,...,...
7484,2020-03-07,Southampton,7
7485,2020-03-07,Wolves,6
7486,2020-03-08,Chelsea,8
7487,2020-03-08,Man United,7


In [46]:
rex3 = pd.merge(rex1,rex2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index(),on=['Date','HomeTeam'])

In [47]:
rex3 = rex3.loc[rex3['menha_date_x'] == rex3['menha_date_y']]

In [48]:
rex3[['Date','Date_'+str(rolling_num)+'Away_of_home_sum','menha_date_y','menha_date_x']]

,Date,Date_10Away_of_home_sum,menha_date_y,menha_date_x
0,2000-08-21,2000-08-19,2,2
374,2000-08-22,2000-08-19,3,3
412,2000-08-22,2000-08-19,3,3
602,2000-08-22,2000-08-19,3,3
621,2000-08-23,2000-08-19,4,4
...,...,...,...,...
1929965,2020-03-07,2020-02-01,35,35
1930074,2020-03-07,2020-02-29,7,7
1930449,2020-03-08,2020-02-29,8,8
1930824,2020-03-08,2020-03-01,7,7


In [49]:
col= rex3.loc[:,'Date_'+str(rolling_num)+'Away_of_home_sum':'menha_date_x'].columns

In [50]:
list(col)

['Date_10Away_of_home_sum',
 'Final_Home_goals_10Away_of_home_sum',
 'Final_Away_goals_10Away_of_home_sum',
 'Half_Time_Home_Goals_10Away_of_home_sum',
 'Half_Time_Away_Goals_10Away_of_home_sum',
 'Half_Time_Result_10Away_of_home_sum',
 'Crowd_Attendance_10Away_of_home_sum',
 'Home_Shots_10Away_of_home_sum',
 'Away_Shots_10Away_of_home_sum',
 'Home_Shots_Target_10Away_of_home_sum',
 'Away_Shots_Target_10Away_of_home_sum',
 'Home_Woodwork_10Away_of_home_sum',
 'Away_Woodwork_10Away_of_home_sum',
 'Home_Corners_10Away_of_home_sum',
 'Away_Corners_10Away_of_home_sum',
 'Home_Fouls_10Away_of_home_sum',
 'Away_Fouls_10Away_of_home_sum',
 'Home_Offsides_10Away_of_home_sum',
 'Away_Offsides_10Away_of_home_sum',
 'Home_Yellow_Cards_10Away_of_home_sum',
 'Away_Yellow_Cards_10Away_of_home_sum',
 'Home_Red_Cards_10Away_of_home_sum',
 'Away_Red_Cards_10Away_of_home_sum',
 'Home_Bookings_Points_10Away_of_home_sum',
 'Away_Bookings_Points_10Away_of_home_sum',
 'Final_result_10Away_of_home_sum',
 'Ho

In [51]:
list(col)+['Date','HomeTeam']

['Date_10Away_of_home_sum',
 'Final_Home_goals_10Away_of_home_sum',
 'Final_Away_goals_10Away_of_home_sum',
 'Half_Time_Home_Goals_10Away_of_home_sum',
 'Half_Time_Away_Goals_10Away_of_home_sum',
 'Half_Time_Result_10Away_of_home_sum',
 'Crowd_Attendance_10Away_of_home_sum',
 'Home_Shots_10Away_of_home_sum',
 'Away_Shots_10Away_of_home_sum',
 'Home_Shots_Target_10Away_of_home_sum',
 'Away_Shots_Target_10Away_of_home_sum',
 'Home_Woodwork_10Away_of_home_sum',
 'Away_Woodwork_10Away_of_home_sum',
 'Home_Corners_10Away_of_home_sum',
 'Away_Corners_10Away_of_home_sum',
 'Home_Fouls_10Away_of_home_sum',
 'Away_Fouls_10Away_of_home_sum',
 'Home_Offsides_10Away_of_home_sum',
 'Away_Offsides_10Away_of_home_sum',
 'Home_Yellow_Cards_10Away_of_home_sum',
 'Away_Yellow_Cards_10Away_of_home_sum',
 'Home_Red_Cards_10Away_of_home_sum',
 'Away_Red_Cards_10Away_of_home_sum',
 'Home_Bookings_Points_10Away_of_home_sum',
 'Away_Bookings_Points_10Away_of_home_sum',
 'Final_result_10Away_of_home_sum',
 'Ho

In [52]:
rex3= rex3.loc[:,list(col)+['Date','HomeTeam']]

In [53]:
rex3 = rex3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Away_of_home_sum'})

In [54]:
# Rolling (median) df2 on 'AwayTeam' and merge with df2 only on "awayteam" without date
nat1= pd.merge(df2, df2.set_index('Date').groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).median().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Away_of_home_median'], left_on=['HomeTeam'],right_on=['AwayTeam'])

In [55]:
nat1['menha_date']= (pd.to_datetime(nat1['Date'])-pd.to_datetime(nat1['Date_'+str(rolling_num)+'Away_of_home_median'])).dt.days

In [56]:
filt= nat1['menha_date']>0
nat2 = nat1.loc[filt]

In [57]:
nat2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index()

,Date,HomeTeam,menha_date
0,2000-08-21,Arsenal,2
1,2000-08-22,Bradford,3
2,2000-08-22,Ipswich,3
3,2000-08-22,Middlesbrough,3
4,2000-08-23,Everton,4
...,...,...,...
7484,2020-03-07,Southampton,7
7485,2020-03-07,Wolves,6
7486,2020-03-08,Chelsea,8
7487,2020-03-08,Man United,7


In [58]:
nat3 = pd.merge(nat1,nat2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index(),on=['Date','HomeTeam'])

In [59]:
nat3= nat3.loc[nat3['menha_date_x']==nat3['menha_date_y']]
nat3.tail()

,Date,HomeTeam,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score,AwayTeam_10Away_of_home_median,Date_10Away_of_home_median,Final_Home_goals_10Away_of_home_median,Final_Away_goals_10Away_of_home_median,Half_Time_Home_Goals_10Away_of_home_median,Half_Time_Away_Goals_10Away_of_home_median,Half_Time_Result_10Away_of_home_median,Crowd_Attendance_10Away_of_home_median,Home_Shots_10Away_of_home_median,Away_Shots_10Away_of_home_median,Home_Shots_Target_10Away_of_home_median,Away_Shots_Target_10Away_of_home_median,Home_Woodwork_10Away_of_home_median,Away_Woodwork_10Away_of_home_median,Home_Corners_10Away_of_home_median,Away_Corners_10Away_of_home_median,Home_Fouls_10Away_of_home_median,Away_Fouls_10Away_of_home_median,Home_Offsides_10Away_of_home_median,Away_Offsides_10Away_of_home_median,Home_Yellow_Cards_10Away_of_home_median,Away_Yellow_Cards_10Away_of_home_median,Home_Red_Cards_10Away_of_home_median,Away_Red_Cards_10Away_of_home_median,Home_Bookings_Points_10Away_of_home_median,Away_Bookings_Points_10Away_of_home_median,Final_result_10Away_of_home_median,Home_score_10Away_of_home_median,Away_score_10Away_of_home_median,menha_date_x,menha_date_y
1929965,2020-03-07,Sheffield United,Norwich,1.0,0.0,1.0,0.0,1,NaN,S Hooper,10.0,12.0,4.0,5.0,NaN,NaN,10.0,5.0,12.0,8.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Sheffield United,2020-02-01,1.0,1.0,0.0,0.0,2.0,NaN,13.0,9.0,4.0,3.0,NaN,NaN,7.5,4.0,6.5,10.5,NaN,NaN,1.0,2.0,0.0,0.0,NaN,NaN,2.0,1.0,1.0,35,35
1930074,2020-03-07,Burnley,Tottenham,1.0,1.0,1.0,0.0,1,NaN,J Moss,21.0,13.0,8.0,2.0,NaN,NaN,3.0,5.0,16.0,11.0,NaN,NaN,5.0,4.0,0.0,0.0,NaN,NaN,2.0,1.0,1.0,Burnley,2020-02-29,1.0,0.5,0.5,0.0,2.0,NaN,16.5,7.0,4.5,1.5,NaN,NaN,7.0,3.5,11.0,10.5,NaN,NaN,0.5,2.5,0.0,0.0,NaN,NaN,1.5,2.0,0.5,7,7
1930449,2020-03-08,Chelsea,Everton,4.0,0.0,2.0,0.0,1,NaN,K Friend,17.0,3.0,11.0,1.0,NaN,NaN,6.0,1.0,8.0,10.0,NaN,NaN,1.0,2.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Chelsea,2020-02-29,1.5,2.0,0.0,1.0,2.5,NaN,12.5,15.5,4.0,4.0,NaN,NaN,3.5,5.5,8.5,10.0,NaN,NaN,2.5,2.0,0.0,0.0,NaN,NaN,2.0,1.0,1.0,8,8
1930824,2020-03-08,Man United,Man City,2.0,0.0,1.0,0.0,1,NaN,M Dean,12.0,7.0,6.0,2.0,NaN,NaN,2.0,11.0,11.0,9.0,NaN,NaN,2.0,4.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Man United,2020-03-01,1.0,1.5,0.5,0.5,2.0,NaN,12.0,12.5,4.5,5.0,NaN,NaN,5.5,5.0,12.0,10.5,NaN,NaN,3.0,2.5,0.0,0.0,NaN,NaN,2.0,1.0,1.0,7,7
1930990,2020-03-09,Leicester,Aston Villa,4.0,0.0,1.0,0.0,1,NaN,M Oliver,15.0,4.0,7.0,1.0,NaN,NaN,9.0,0.0,15.0,12.0,NaN,NaN,2.0,1.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Leicester,2020-02-28,0.5,2.0,0.0,1.0,2.0,NaN,11.5,17.5,3.0,8.0,NaN,NaN,5.5,5.0,13.5,11.0,NaN,NaN,2.0,1.5,0.0,0.0,NaN,NaN,3.0,0.0,3.0,10,10


In [60]:
col= nat3.loc[:,'Date_'+str(rolling_num)+'Away_of_home_median':'menha_date_x'].columns

In [61]:
list(col)

['Date_10Away_of_home_median',
 'Final_Home_goals_10Away_of_home_median',
 'Final_Away_goals_10Away_of_home_median',
 'Half_Time_Home_Goals_10Away_of_home_median',
 'Half_Time_Away_Goals_10Away_of_home_median',
 'Half_Time_Result_10Away_of_home_median',
 'Crowd_Attendance_10Away_of_home_median',
 'Home_Shots_10Away_of_home_median',
 'Away_Shots_10Away_of_home_median',
 'Home_Shots_Target_10Away_of_home_median',
 'Away_Shots_Target_10Away_of_home_median',
 'Home_Woodwork_10Away_of_home_median',
 'Away_Woodwork_10Away_of_home_median',
 'Home_Corners_10Away_of_home_median',
 'Away_Corners_10Away_of_home_median',
 'Home_Fouls_10Away_of_home_median',
 'Away_Fouls_10Away_of_home_median',
 'Home_Offsides_10Away_of_home_median',
 'Away_Offsides_10Away_of_home_median',
 'Home_Yellow_Cards_10Away_of_home_median',
 'Away_Yellow_Cards_10Away_of_home_median',
 'Home_Red_Cards_10Away_of_home_median',
 'Away_Red_Cards_10Away_of_home_median',
 'Home_Bookings_Points_10Away_of_home_median',
 'Away_Booki

In [62]:
['Date','HomeTeam']+ list(col)

['Date',
 'HomeTeam',
 'Date_10Away_of_home_median',
 'Final_Home_goals_10Away_of_home_median',
 'Final_Away_goals_10Away_of_home_median',
 'Half_Time_Home_Goals_10Away_of_home_median',
 'Half_Time_Away_Goals_10Away_of_home_median',
 'Half_Time_Result_10Away_of_home_median',
 'Crowd_Attendance_10Away_of_home_median',
 'Home_Shots_10Away_of_home_median',
 'Away_Shots_10Away_of_home_median',
 'Home_Shots_Target_10Away_of_home_median',
 'Away_Shots_Target_10Away_of_home_median',
 'Home_Woodwork_10Away_of_home_median',
 'Away_Woodwork_10Away_of_home_median',
 'Home_Corners_10Away_of_home_median',
 'Away_Corners_10Away_of_home_median',
 'Home_Fouls_10Away_of_home_median',
 'Away_Fouls_10Away_of_home_median',
 'Home_Offsides_10Away_of_home_median',
 'Away_Offsides_10Away_of_home_median',
 'Home_Yellow_Cards_10Away_of_home_median',
 'Away_Yellow_Cards_10Away_of_home_median',
 'Home_Red_Cards_10Away_of_home_median',
 'Away_Red_Cards_10Away_of_home_median',
 'Home_Bookings_Points_10Away_of_home

In [63]:
nat3= nat3.loc[:,['Date','HomeTeam']+ list(col)]

In [64]:
nat3 = nat3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Away_of_home_median'})

In [65]:
# Rolling (mean) df2 on 'AwayTeam' and merge with df2 only on "awayteam" without date
jav1= pd.merge(df2, df2.set_index('Date').groupby(['AwayTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Away_of_home_mean'], left_on=['HomeTeam'],right_on=['AwayTeam'])

In [66]:
jav1['menha_date']= (pd.to_datetime(jav1['Date'])-pd.to_datetime(jav1['Date_'+str(rolling_num)+'Away_of_home_mean'])).dt.days

In [67]:
filt= jav1['menha_date']>0
jav2 = jav1.loc[filt]

In [68]:
jav2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index()

,Date,HomeTeam,menha_date
0,2000-08-21,Arsenal,2
1,2000-08-22,Bradford,3
2,2000-08-22,Ipswich,3
3,2000-08-22,Middlesbrough,3
4,2000-08-23,Everton,4
...,...,...,...
7484,2020-03-07,Southampton,7
7485,2020-03-07,Wolves,6
7486,2020-03-08,Chelsea,8
7487,2020-03-08,Man United,7


In [69]:
jav3 = pd.merge(jav1,jav2.groupby(['Date','HomeTeam'])['menha_date'].min().reset_index(),on=['Date','HomeTeam'])

In [70]:
jav3 = jav3.loc[jav3['menha_date_x'] == jav3['menha_date_y']]

In [71]:
jav3[['Date','Date_'+str(rolling_num)+'Away_of_home_mean','menha_date_y','menha_date_x']]

,Date,Date_10Away_of_home_mean,menha_date_y,menha_date_x
0,2000-08-21,2000-08-19,2,2
374,2000-08-22,2000-08-19,3,3
412,2000-08-22,2000-08-19,3,3
602,2000-08-22,2000-08-19,3,3
621,2000-08-23,2000-08-19,4,4
...,...,...,...,...
1929965,2020-03-07,2020-02-01,35,35
1930074,2020-03-07,2020-02-29,7,7
1930449,2020-03-08,2020-02-29,8,8
1930824,2020-03-08,2020-03-01,7,7


In [72]:
col= jav3.loc[:,'Date_'+str(rolling_num)+'Away_of_home_mean':'menha_date_x'].columns

In [73]:
list(col)

['Date_10Away_of_home_mean',
 'Final_Home_goals_10Away_of_home_mean',
 'Final_Away_goals_10Away_of_home_mean',
 'Half_Time_Home_Goals_10Away_of_home_mean',
 'Half_Time_Away_Goals_10Away_of_home_mean',
 'Half_Time_Result_10Away_of_home_mean',
 'Crowd_Attendance_10Away_of_home_mean',
 'Home_Shots_10Away_of_home_mean',
 'Away_Shots_10Away_of_home_mean',
 'Home_Shots_Target_10Away_of_home_mean',
 'Away_Shots_Target_10Away_of_home_mean',
 'Home_Woodwork_10Away_of_home_mean',
 'Away_Woodwork_10Away_of_home_mean',
 'Home_Corners_10Away_of_home_mean',
 'Away_Corners_10Away_of_home_mean',
 'Home_Fouls_10Away_of_home_mean',
 'Away_Fouls_10Away_of_home_mean',
 'Home_Offsides_10Away_of_home_mean',
 'Away_Offsides_10Away_of_home_mean',
 'Home_Yellow_Cards_10Away_of_home_mean',
 'Away_Yellow_Cards_10Away_of_home_mean',
 'Home_Red_Cards_10Away_of_home_mean',
 'Away_Red_Cards_10Away_of_home_mean',
 'Home_Bookings_Points_10Away_of_home_mean',
 'Away_Bookings_Points_10Away_of_home_mean',
 'Final_result_

In [74]:
['Date','HomeTeam']+ list(col)

['Date',
 'HomeTeam',
 'Date_10Away_of_home_mean',
 'Final_Home_goals_10Away_of_home_mean',
 'Final_Away_goals_10Away_of_home_mean',
 'Half_Time_Home_Goals_10Away_of_home_mean',
 'Half_Time_Away_Goals_10Away_of_home_mean',
 'Half_Time_Result_10Away_of_home_mean',
 'Crowd_Attendance_10Away_of_home_mean',
 'Home_Shots_10Away_of_home_mean',
 'Away_Shots_10Away_of_home_mean',
 'Home_Shots_Target_10Away_of_home_mean',
 'Away_Shots_Target_10Away_of_home_mean',
 'Home_Woodwork_10Away_of_home_mean',
 'Away_Woodwork_10Away_of_home_mean',
 'Home_Corners_10Away_of_home_mean',
 'Away_Corners_10Away_of_home_mean',
 'Home_Fouls_10Away_of_home_mean',
 'Away_Fouls_10Away_of_home_mean',
 'Home_Offsides_10Away_of_home_mean',
 'Away_Offsides_10Away_of_home_mean',
 'Home_Yellow_Cards_10Away_of_home_mean',
 'Away_Yellow_Cards_10Away_of_home_mean',
 'Home_Red_Cards_10Away_of_home_mean',
 'Away_Red_Cards_10Away_of_home_mean',
 'Home_Bookings_Points_10Away_of_home_mean',
 'Away_Bookings_Points_10Away_of_home_

In [75]:
jav3= jav3.loc[:,['Date','HomeTeam']+ list(col)]

In [76]:
jav3 = jav3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Away_of_home_mean'})

In [77]:
df_f= pd.merge(rex3, nat3, on=['HomeTeam','Date'])

In [78]:
df_3Away_of_home= pd.merge(df_f, jav3, on=['HomeTeam','Date'])

In [79]:
# Rolling (sum) df2 on 'HomeTeam' and merge with df2 only on "hometeam" without date
erf1= pd.merge(df2, df2.set_index('Date').groupby(['HomeTeam']).rolling(rolling_num, min_periods=rolling_num).sum().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Home_of_away_sum'], left_on=['AwayTeam'],right_on=['HomeTeam'])

In [80]:
erf1['menha_date']= (pd.to_datetime(erf1['Date'])-pd.to_datetime(erf1['Date_'+str(rolling_num)+'Home_of_away_sum'])).dt.days

In [81]:
filt= erf1['menha_date']>0
erf2 = erf1.loc[filt]

In [82]:
erf2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index()

,Date,AwayTeam,menha_date
0,2000-08-21,Liverpool,2
1,2000-08-22,Chelsea,3
2,2000-08-22,Man United,2
3,2000-08-22,Tottenham,3
4,2000-08-23,Charlton,4
...,...,...,...
7478,2020-03-07,Watford,7
7479,2020-03-07,West Ham,7
7480,2020-03-08,Everton,7
7481,2020-03-08,Man City,18


In [83]:
erf3 = pd.merge(erf1,erf2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index(),on=['Date','AwayTeam'])

In [84]:
erf3 = erf3.loc[erf3['menha_date_x'] == erf3['menha_date_y']]

In [85]:
erf3[['Date','Date_'+str(rolling_num)+'Home_of_away_sum','menha_date_y','menha_date_x']]

,Date,Date_10Home_of_away_sum,menha_date_y,menha_date_x
0,2000-08-21,2000-08-19,2,2
376,2000-08-22,2000-08-20,2,2
752,2000-08-22,2000-08-19,3,3
1127,2000-08-22,2000-08-19,3,3
1503,2000-08-23,2000-08-19,4,4
...,...,...,...,...
1928039,2020-03-07,2020-02-28,8,8
1928414,2020-03-07,2020-03-01,6,6
1928789,2020-03-08,2020-03-01,7,7
1929144,2020-03-08,2020-02-19,18,18


In [86]:
col= erf3.loc[:,'Date_'+str(rolling_num)+'Home_of_away_sum':'menha_date_x'].columns

In [87]:
list(col)

['Date_10Home_of_away_sum',
 'Final_Home_goals_10Home_of_away_sum',
 'Final_Away_goals_10Home_of_away_sum',
 'Half_Time_Home_Goals_10Home_of_away_sum',
 'Half_Time_Away_Goals_10Home_of_away_sum',
 'Half_Time_Result_10Home_of_away_sum',
 'Crowd_Attendance_10Home_of_away_sum',
 'Home_Shots_10Home_of_away_sum',
 'Away_Shots_10Home_of_away_sum',
 'Home_Shots_Target_10Home_of_away_sum',
 'Away_Shots_Target_10Home_of_away_sum',
 'Home_Woodwork_10Home_of_away_sum',
 'Away_Woodwork_10Home_of_away_sum',
 'Home_Corners_10Home_of_away_sum',
 'Away_Corners_10Home_of_away_sum',
 'Home_Fouls_10Home_of_away_sum',
 'Away_Fouls_10Home_of_away_sum',
 'Home_Offsides_10Home_of_away_sum',
 'Away_Offsides_10Home_of_away_sum',
 'Home_Yellow_Cards_10Home_of_away_sum',
 'Away_Yellow_Cards_10Home_of_away_sum',
 'Home_Red_Cards_10Home_of_away_sum',
 'Away_Red_Cards_10Home_of_away_sum',
 'Home_Bookings_Points_10Home_of_away_sum',
 'Away_Bookings_Points_10Home_of_away_sum',
 'Final_result_10Home_of_away_sum',
 'Ho

In [88]:
['Date','AwayTeam']+ list(col)

['Date',
 'AwayTeam',
 'Date_10Home_of_away_sum',
 'Final_Home_goals_10Home_of_away_sum',
 'Final_Away_goals_10Home_of_away_sum',
 'Half_Time_Home_Goals_10Home_of_away_sum',
 'Half_Time_Away_Goals_10Home_of_away_sum',
 'Half_Time_Result_10Home_of_away_sum',
 'Crowd_Attendance_10Home_of_away_sum',
 'Home_Shots_10Home_of_away_sum',
 'Away_Shots_10Home_of_away_sum',
 'Home_Shots_Target_10Home_of_away_sum',
 'Away_Shots_Target_10Home_of_away_sum',
 'Home_Woodwork_10Home_of_away_sum',
 'Away_Woodwork_10Home_of_away_sum',
 'Home_Corners_10Home_of_away_sum',
 'Away_Corners_10Home_of_away_sum',
 'Home_Fouls_10Home_of_away_sum',
 'Away_Fouls_10Home_of_away_sum',
 'Home_Offsides_10Home_of_away_sum',
 'Away_Offsides_10Home_of_away_sum',
 'Home_Yellow_Cards_10Home_of_away_sum',
 'Away_Yellow_Cards_10Home_of_away_sum',
 'Home_Red_Cards_10Home_of_away_sum',
 'Away_Red_Cards_10Home_of_away_sum',
 'Home_Bookings_Points_10Home_of_away_sum',
 'Away_Bookings_Points_10Home_of_away_sum',
 'Final_result_10H

In [89]:
erf3= erf3.loc[:,['Date','AwayTeam']+ list(col)]

In [90]:
erf3 = erf3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Home_of_away_sum'})

In [91]:
# Rolling (median) df2 on 'HomeTeam' and merge with df2 only on "hometeam" without date
eli1= pd.merge(df2, df2.set_index('Date').groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).median().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Home_of_away_median'], left_on=['AwayTeam'],right_on=['HomeTeam'])

In [92]:
eli1['menha_date']= (pd.to_datetime(eli1['Date'])-pd.to_datetime(eli1['Date_'+str(rolling_num)+'Home_of_away_median'])).dt.days

In [93]:
filt= eli1['menha_date']>0
eli2 = eli1.loc[filt]

In [94]:
eli2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index()

,Date,AwayTeam,menha_date
0,2000-08-21,Liverpool,2
1,2000-08-22,Chelsea,3
2,2000-08-22,Man United,2
3,2000-08-22,Tottenham,3
4,2000-08-23,Charlton,4
...,...,...,...
7478,2020-03-07,Watford,7
7479,2020-03-07,West Ham,7
7480,2020-03-08,Everton,7
7481,2020-03-08,Man City,18


In [95]:
eli3 = pd.merge(eli1,eli2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index(),on=['Date','AwayTeam'])

In [96]:
eli3= eli3.loc[eli3['menha_date_x']==eli3['menha_date_y']]
eli3.tail()

,Date,HomeTeam,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score,HomeTeam_10Home_of_away_median,Date_10Home_of_away_median,Final_Home_goals_10Home_of_away_median,Final_Away_goals_10Home_of_away_median,Half_Time_Home_Goals_10Home_of_away_median,Half_Time_Away_Goals_10Home_of_away_median,Half_Time_Result_10Home_of_away_median,Crowd_Attendance_10Home_of_away_median,Home_Shots_10Home_of_away_median,Away_Shots_10Home_of_away_median,Home_Shots_Target_10Home_of_away_median,Away_Shots_Target_10Home_of_away_median,Home_Woodwork_10Home_of_away_median,Away_Woodwork_10Home_of_away_median,Home_Corners_10Home_of_away_median,Away_Corners_10Home_of_away_median,Home_Fouls_10Home_of_away_median,Away_Fouls_10Home_of_away_median,Home_Offsides_10Home_of_away_median,Away_Offsides_10Home_of_away_median,Home_Yellow_Cards_10Home_of_away_median,Away_Yellow_Cards_10Home_of_away_median,Home_Red_Cards_10Home_of_away_median,Away_Red_Cards_10Home_of_away_median,Home_Bookings_Points_10Home_of_away_median,Away_Bookings_Points_10Home_of_away_median,Final_result_10Home_of_away_median,Home_score_10Home_of_away_median,Away_score_10Home_of_away_median,menha_date_x,menha_date_y
1928039,2020-03-07,Sheffield United,Norwich,1.0,0.0,1.0,0.0,1,NaN,S Hooper,10.0,12.0,4.0,5.0,NaN,NaN,10.0,5.0,12.0,8.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Norwich,2020-02-28,1.0,2.0,1.0,0.0,1.0,NaN,13.5,14.5,4.5,4.0,NaN,NaN,4.0,6.5,9.0,10.5,NaN,NaN,2.0,2.0,0.0,0.0,NaN,NaN,2.5,0.5,2.0,8,8
1928414,2020-03-07,Burnley,Tottenham,1.0,1.0,1.0,0.0,1,NaN,J Moss,21.0,13.0,8.0,2.0,NaN,NaN,3.0,5.0,16.0,11.0,NaN,NaN,5.0,4.0,0.0,0.0,NaN,NaN,2.0,1.0,1.0,Tottenham,2020-03-01,2.0,1.0,0.0,0.5,2.0,NaN,13.0,13.0,4.5,4.0,NaN,NaN,4.0,3.5,9.5,9.0,NaN,NaN,2.0,2.0,0.0,0.0,NaN,NaN,1.5,2.0,0.5,6,6
1928789,2020-03-08,Chelsea,Everton,4.0,0.0,2.0,0.0,1,NaN,K Friend,17.0,3.0,11.0,1.0,NaN,NaN,6.0,1.0,8.0,10.0,NaN,NaN,1.0,2.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Everton,2020-03-01,1.0,1.0,1.0,0.0,1.5,NaN,15.0,8.5,6.5,2.0,NaN,NaN,5.5,4.0,11.5,11.0,NaN,NaN,2.0,1.5,0.0,0.0,NaN,NaN,1.5,2.0,0.5,7,7
1929144,2020-03-08,Man United,Man City,2.0,0.0,1.0,0.0,1,NaN,M Dean,12.0,7.0,6.0,2.0,NaN,NaN,2.0,11.0,11.0,9.0,NaN,NaN,2.0,4.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Man City,2020-02-19,2.0,1.0,0.0,0.5,2.0,NaN,21.0,7.0,5.5,2.0,NaN,NaN,8.5,2.0,10.0,9.0,NaN,NaN,1.0,1.5,0.0,0.0,NaN,NaN,1.0,3.0,0.0,18,18
1929461,2020-03-09,Leicester,Aston Villa,4.0,0.0,1.0,0.0,1,NaN,M Oliver,15.0,4.0,7.0,1.0,NaN,NaN,9.0,0.0,15.0,12.0,NaN,NaN,2.0,1.0,0.0,0.0,NaN,NaN,1.0,3.0,0.0,Aston Villa,2020-02-16,1.5,2.0,1.0,1.0,2.5,NaN,16.0,18.5,3.5,6.0,NaN,NaN,7.0,5.0,11.5,13.5,NaN,NaN,2.0,2.0,0.0,0.0,NaN,NaN,2.5,0.5,2.0,22,22


In [97]:
col= eli3.loc[:,'Date_'+str(rolling_num)+'Home_of_away_median':'menha_date_x'].columns

In [98]:
list(col)

['Date_10Home_of_away_median',
 'Final_Home_goals_10Home_of_away_median',
 'Final_Away_goals_10Home_of_away_median',
 'Half_Time_Home_Goals_10Home_of_away_median',
 'Half_Time_Away_Goals_10Home_of_away_median',
 'Half_Time_Result_10Home_of_away_median',
 'Crowd_Attendance_10Home_of_away_median',
 'Home_Shots_10Home_of_away_median',
 'Away_Shots_10Home_of_away_median',
 'Home_Shots_Target_10Home_of_away_median',
 'Away_Shots_Target_10Home_of_away_median',
 'Home_Woodwork_10Home_of_away_median',
 'Away_Woodwork_10Home_of_away_median',
 'Home_Corners_10Home_of_away_median',
 'Away_Corners_10Home_of_away_median',
 'Home_Fouls_10Home_of_away_median',
 'Away_Fouls_10Home_of_away_median',
 'Home_Offsides_10Home_of_away_median',
 'Away_Offsides_10Home_of_away_median',
 'Home_Yellow_Cards_10Home_of_away_median',
 'Away_Yellow_Cards_10Home_of_away_median',
 'Home_Red_Cards_10Home_of_away_median',
 'Away_Red_Cards_10Home_of_away_median',
 'Home_Bookings_Points_10Home_of_away_median',
 'Away_Booki

In [99]:
['Date','AwayTeam']+ list(col)

['Date',
 'AwayTeam',
 'Date_10Home_of_away_median',
 'Final_Home_goals_10Home_of_away_median',
 'Final_Away_goals_10Home_of_away_median',
 'Half_Time_Home_Goals_10Home_of_away_median',
 'Half_Time_Away_Goals_10Home_of_away_median',
 'Half_Time_Result_10Home_of_away_median',
 'Crowd_Attendance_10Home_of_away_median',
 'Home_Shots_10Home_of_away_median',
 'Away_Shots_10Home_of_away_median',
 'Home_Shots_Target_10Home_of_away_median',
 'Away_Shots_Target_10Home_of_away_median',
 'Home_Woodwork_10Home_of_away_median',
 'Away_Woodwork_10Home_of_away_median',
 'Home_Corners_10Home_of_away_median',
 'Away_Corners_10Home_of_away_median',
 'Home_Fouls_10Home_of_away_median',
 'Away_Fouls_10Home_of_away_median',
 'Home_Offsides_10Home_of_away_median',
 'Away_Offsides_10Home_of_away_median',
 'Home_Yellow_Cards_10Home_of_away_median',
 'Away_Yellow_Cards_10Home_of_away_median',
 'Home_Red_Cards_10Home_of_away_median',
 'Away_Red_Cards_10Home_of_away_median',
 'Home_Bookings_Points_10Home_of_away

In [100]:
eli3= eli3.loc[:,['Date','AwayTeam']+ list(col)]

In [101]:
eli3 = eli3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Home_of_away_median'})

In [102]:
# Rolling (mean) df2 on 'HomeTeam' and merge with df2 only on "hometeam" without date
bah1= pd.merge(df2, df2.set_index('Date').groupby(['HomeTeam']).rolling(rolling_num, min_periods=1).mean().reset_index(),  how='left', suffixes=['','_'+str(rolling_num)+'Home_of_away_mean'], left_on=['AwayTeam'],right_on=['HomeTeam'])

In [103]:
bah1['menha_date']= (pd.to_datetime(bah1['Date'])-pd.to_datetime(bah1['Date_'+str(rolling_num)+'Home_of_away_mean'])).dt.days

In [104]:
filt= bah1['menha_date']>0
bah2 = bah1.loc[filt]

In [105]:
bah2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index()

,Date,AwayTeam,menha_date
0,2000-08-21,Liverpool,2
1,2000-08-22,Chelsea,3
2,2000-08-22,Man United,2
3,2000-08-22,Tottenham,3
4,2000-08-23,Charlton,4
...,...,...,...
7478,2020-03-07,Watford,7
7479,2020-03-07,West Ham,7
7480,2020-03-08,Everton,7
7481,2020-03-08,Man City,18


In [106]:
bah3 = pd.merge(bah1,bah2.groupby(['Date','AwayTeam'])['menha_date'].min().reset_index(),on=['Date','AwayTeam'])

In [107]:
bah3 = bah3.loc[bah3['menha_date_x'] == bah3['menha_date_y']]

In [108]:
bah3[['Date','Date_'+str(rolling_num)+'Home_of_away_mean','menha_date_y','menha_date_x']]

,Date,Date_10Home_of_away_mean,menha_date_y,menha_date_x
0,2000-08-21,2000-08-19,2,2
376,2000-08-22,2000-08-20,2,2
752,2000-08-22,2000-08-19,3,3
1127,2000-08-22,2000-08-19,3,3
1503,2000-08-23,2000-08-19,4,4
...,...,...,...,...
1928039,2020-03-07,2020-02-28,8,8
1928414,2020-03-07,2020-03-01,6,6
1928789,2020-03-08,2020-03-01,7,7
1929144,2020-03-08,2020-02-19,18,18


In [109]:
col= bah3.loc[:,'Date_'+str(rolling_num)+'Home_of_away_mean':'menha_date_x'].columns

In [110]:
list(col)

['Date_10Home_of_away_mean',
 'Final_Home_goals_10Home_of_away_mean',
 'Final_Away_goals_10Home_of_away_mean',
 'Half_Time_Home_Goals_10Home_of_away_mean',
 'Half_Time_Away_Goals_10Home_of_away_mean',
 'Half_Time_Result_10Home_of_away_mean',
 'Crowd_Attendance_10Home_of_away_mean',
 'Home_Shots_10Home_of_away_mean',
 'Away_Shots_10Home_of_away_mean',
 'Home_Shots_Target_10Home_of_away_mean',
 'Away_Shots_Target_10Home_of_away_mean',
 'Home_Woodwork_10Home_of_away_mean',
 'Away_Woodwork_10Home_of_away_mean',
 'Home_Corners_10Home_of_away_mean',
 'Away_Corners_10Home_of_away_mean',
 'Home_Fouls_10Home_of_away_mean',
 'Away_Fouls_10Home_of_away_mean',
 'Home_Offsides_10Home_of_away_mean',
 'Away_Offsides_10Home_of_away_mean',
 'Home_Yellow_Cards_10Home_of_away_mean',
 'Away_Yellow_Cards_10Home_of_away_mean',
 'Home_Red_Cards_10Home_of_away_mean',
 'Away_Red_Cards_10Home_of_away_mean',
 'Home_Bookings_Points_10Home_of_away_mean',
 'Away_Bookings_Points_10Home_of_away_mean',
 'Final_result_

In [111]:
['Date','AwayTeam']+ list(col)

['Date',
 'AwayTeam',
 'Date_10Home_of_away_mean',
 'Final_Home_goals_10Home_of_away_mean',
 'Final_Away_goals_10Home_of_away_mean',
 'Half_Time_Home_Goals_10Home_of_away_mean',
 'Half_Time_Away_Goals_10Home_of_away_mean',
 'Half_Time_Result_10Home_of_away_mean',
 'Crowd_Attendance_10Home_of_away_mean',
 'Home_Shots_10Home_of_away_mean',
 'Away_Shots_10Home_of_away_mean',
 'Home_Shots_Target_10Home_of_away_mean',
 'Away_Shots_Target_10Home_of_away_mean',
 'Home_Woodwork_10Home_of_away_mean',
 'Away_Woodwork_10Home_of_away_mean',
 'Home_Corners_10Home_of_away_mean',
 'Away_Corners_10Home_of_away_mean',
 'Home_Fouls_10Home_of_away_mean',
 'Away_Fouls_10Home_of_away_mean',
 'Home_Offsides_10Home_of_away_mean',
 'Away_Offsides_10Home_of_away_mean',
 'Home_Yellow_Cards_10Home_of_away_mean',
 'Away_Yellow_Cards_10Home_of_away_mean',
 'Home_Red_Cards_10Home_of_away_mean',
 'Away_Red_Cards_10Home_of_away_mean',
 'Home_Bookings_Points_10Home_of_away_mean',
 'Away_Bookings_Points_10Home_of_away_

In [112]:
bah3= bah3.loc[:,['Date','AwayTeam']+ list(col)]

In [113]:
bah3 = bah3.rename(columns = {'menha_date_x':'menha_date_'+str(rolling_num)+'Home_of_away_mean'})

In [114]:
df_f_1= pd.merge(erf3, eli3, on=['AwayTeam','Date'])

In [115]:
df_3Home_of_away= pd.merge(df_f_1, bah3, on=['AwayTeam','Date'])

In [116]:
df_naha= pd.merge(df_final, df_3Away_of_home, how='left', on=['Date','HomeTeam'])
df_naha

,Date,HomeTeam,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score,Final_Home_goals_HomeTeam_sum_10,Final_Away_goals_HomeTeam_sum_10,Half_Time_Home_Goals_HomeTeam_sum_10,Half_Time_Away_Goals_HomeTeam_sum_10,Half_Time_Result_HomeTeam_sum_10,Crowd_Attendance_HomeTeam_sum_10,Home_Shots_HomeTeam_sum_10,Away_Shots_HomeTeam_sum_10,Home_Shots_Target_HomeTeam_sum_10,Away_Shots_Target_HomeTeam_sum_10,Home_Woodwork_HomeTeam_sum_10,Away_Woodwork_HomeTeam_sum_10,Home_Corners_HomeTeam_sum_10,Away_Corners_HomeTeam_sum_10,Home_Fouls_HomeTeam_sum_10,Away_Fouls_HomeTeam_sum_10,Home_Offsides_HomeTeam_sum_10,Away_Offsides_HomeTeam_sum_10,Home_Yellow_Cards_HomeTeam_sum_10,Away_Yellow_Cards_HomeTeam_sum_10,Home_Red_Cards_HomeTeam_sum_10,Away_Red_Cards_HomeTeam_sum_10,Home_Bookings_Points_HomeTeam_sum_10,Away_Bookings_Points_HomeTeam_sum_10,Final_result_HomeTeam_sum_10,Home_score_HomeTeam_sum_10,Away_score_HomeTeam_sum_10,Final_Home_goals_AwayTeam_sum_10,Final_Away_goals_AwayTeam_sum_10,Half_Time_Home_Goals_AwayTeam_sum_10,Half_Time_Away_Goals_AwayTeam_sum_10,Half_Time_Result_AwayTeam_sum_10,Crowd_Attendance_AwayTeam_sum_10,Home_Shots_AwayTeam_sum_10,Away_Shots_AwayTeam_sum_10,Home_Shots_Target_AwayTeam_sum_10,Away_Shots_Target_AwayTeam_sum_10,Home_Woodwork_AwayTeam_sum_10,Away_Woodwork_AwayTeam_sum_10,Home_Corners_AwayTeam_sum_10,Away_Corners_AwayTeam_sum_10,Home_Fouls_AwayTeam_sum_10,Away_Fouls_AwayTeam_sum_10,Home_Offsides_AwayTeam_sum_10,Away_Offsides_AwayTeam_sum_10,Home_Yellow_Cards_AwayTeam_sum_10,Away_Yellow_Cards_AwayTeam_sum_10,Home_Red_Cards_AwayTeam_sum_10,Away_Red_Cards_AwayTeam_sum_10,Home_Bookings_Points_AwayTeam_sum_10,Away_Bookings_Points_AwayTeam_sum_10,Final_result_AwayTeam_sum_10,Home_score_AwayTeam_sum_10,Away_score_AwayTeam_sum_10,Final_Home_goals_HomeTeam_median_10,Final_Away_goals_HomeTeam_median_10,Half_Time_Home_Goals_HomeTeam_median_10,Half_Time_Away_Goals_HomeTeam_median_10,Half_Time_Result_HomeTeam_median_10,Crowd_Attendance_HomeTeam_median_10,Home_Shots_HomeTeam_median_10,Away_Shots_HomeTeam_median_10,Home_Shots_Target_HomeTeam_median_10,Away_Shots_Target_HomeTeam_median_10,Home_Woodwork_HomeTeam_median_10,Away_Woodwork_HomeTeam_median_10,Home_Corners_HomeTeam_median_10,Away_Corners_HomeTeam_median_10,Home_Fouls_HomeTeam_median_10,Away_Fouls_HomeTeam_median_10,Home_Offsides_HomeTeam_median_10,Away_Offsides_HomeTeam_median_10,Home_Yellow_Cards_HomeTeam_median_10,Away_Yellow_Cards_HomeTeam_median_10,Home_Red_Cards_HomeTeam_median_10,Away_Red_Cards_HomeTeam_median_10,Home_Bookings_Points_HomeTeam_median_10,Away_Bookings_Points_HomeTeam_median_10,Final_result_HomeTeam_median_10,Home_score_HomeTeam_median_10,Away_score_HomeTeam_median_10,Final_Home_goals_AwayTeam_median_10,Final_Away_goals_AwayTeam_median_10,Half_Time_Home_Goals_AwayTeam_median_10,Half_Time_Away_Goals_AwayTeam_median_10,Half_Time_Result_AwayTeam_median_10,Crowd_Attendance_AwayTeam_median_10,Home_Shots_AwayTeam_median_10,Away_Shots_AwayTeam_median_10,Home_Shots_Target_AwayTeam_median_10,Away_Shots_Target_AwayTeam_median_10,Home_Woodwork_AwayTeam_median_10,Away_Woodwork_AwayTeam_median_10,Home_Corners_AwayTeam_median_10,Away_Corners_AwayTeam_median_10,Home_Fouls_AwayTeam_median_10,Away_Fouls_AwayTeam_median_10,Home_Offsides_AwayTeam_median_10,Away_Offsides_AwayTeam_median_10,Home_Yellow_Cards_AwayTeam_median_10,Away_Yellow_Cards_AwayTeam_median_10,Home_Red_Cards_AwayTeam_median_10,Away_Red_Cards_AwayTeam_median_10,Home_Bookings_Points_AwayTeam_median_10,Away_Bookings_Points_AwayTeam_median_10,Final_result_AwayTeam_median_10,Home_score_AwayTeam_median_10,Away_score_AwayTeam_median_10,Fin

In [117]:
df_naha= pd.merge(df_naha, df_3Home_of_away, how='left', on=['Date','AwayTeam'])
df_naha

,Date,HomeTeam,AwayTeam,Final_Home_goals,Final_Away_goals,Half_Time_Home_Goals,Half_Time_Away_Goals,Half_Time_Result,Crowd_Attendance,Referee,Home_Shots,Away_Shots,Home_Shots_Target,Away_Shots_Target,Home_Woodwork,Away_Woodwork,Home_Corners,Away_Corners,Home_Fouls,Away_Fouls,Home_Offsides,Away_Offsides,Home_Yellow_Cards,Away_Yellow_Cards,Home_Red_Cards,Away_Red_Cards,Home_Bookings_Points,Away_Bookings_Points,Final_result,Home_score,Away_score,Final_Home_goals_HomeTeam_sum_10,Final_Away_goals_HomeTeam_sum_10,Half_Time_Home_Goals_HomeTeam_sum_10,Half_Time_Away_Goals_HomeTeam_sum_10,Half_Time_Result_HomeTeam_sum_10,Crowd_Attendance_HomeTeam_sum_10,Home_Shots_HomeTeam_sum_10,Away_Shots_HomeTeam_sum_10,Home_Shots_Target_HomeTeam_sum_10,Away_Shots_Target_HomeTeam_sum_10,Home_Woodwork_HomeTeam_sum_10,Away_Woodwork_HomeTeam_sum_10,Home_Corners_HomeTeam_sum_10,Away_Corners_HomeTeam_sum_10,Home_Fouls_HomeTeam_sum_10,Away_Fouls_HomeTeam_sum_10,Home_Offsides_HomeTeam_sum_10,Away_Offsides_HomeTeam_sum_10,Home_Yellow_Cards_HomeTeam_sum_10,Away_Yellow_Cards_HomeTeam_sum_10,Home_Red_Cards_HomeTeam_sum_10,Away_Red_Cards_HomeTeam_sum_10,Home_Bookings_Points_HomeTeam_sum_10,Away_Bookings_Points_HomeTeam_sum_10,Final_result_HomeTeam_sum_10,Home_score_HomeTeam_sum_10,Away_score_HomeTeam_sum_10,Final_Home_goals_AwayTeam_sum_10,Final_Away_goals_AwayTeam_sum_10,Half_Time_Home_Goals_AwayTeam_sum_10,Half_Time_Away_Goals_AwayTeam_sum_10,Half_Time_Result_AwayTeam_sum_10,Crowd_Attendance_AwayTeam_sum_10,Home_Shots_AwayTeam_sum_10,Away_Shots_AwayTeam_sum_10,Home_Shots_Target_AwayTeam_sum_10,Away_Shots_Target_AwayTeam_sum_10,Home_Woodwork_AwayTeam_sum_10,Away_Woodwork_AwayTeam_sum_10,Home_Corners_AwayTeam_sum_10,Away_Corners_AwayTeam_sum_10,Home_Fouls_AwayTeam_sum_10,Away_Fouls_AwayTeam_sum_10,Home_Offsides_AwayTeam_sum_10,Away_Offsides_AwayTeam_sum_10,Home_Yellow_Cards_AwayTeam_sum_10,Away_Yellow_Cards_AwayTeam_sum_10,Home_Red_Cards_AwayTeam_sum_10,Away_Red_Cards_AwayTeam_sum_10,Home_Bookings_Points_AwayTeam_sum_10,Away_Bookings_Points_AwayTeam_sum_10,Final_result_AwayTeam_sum_10,Home_score_AwayTeam_sum_10,Away_score_AwayTeam_sum_10,Final_Home_goals_HomeTeam_median_10,Final_Away_goals_HomeTeam_median_10,Half_Time_Home_Goals_HomeTeam_median_10,Half_Time_Away_Goals_HomeTeam_median_10,Half_Time_Result_HomeTeam_median_10,Crowd_Attendance_HomeTeam_median_10,Home_Shots_HomeTeam_median_10,Away_Shots_HomeTeam_median_10,Home_Shots_Target_HomeTeam_median_10,Away_Shots_Target_HomeTeam_median_10,Home_Woodwork_HomeTeam_median_10,Away_Woodwork_HomeTeam_median_10,Home_Corners_HomeTeam_median_10,Away_Corners_HomeTeam_median_10,Home_Fouls_HomeTeam_median_10,Away_Fouls_HomeTeam_median_10,Home_Offsides_HomeTeam_median_10,Away_Offsides_HomeTeam_median_10,Home_Yellow_Cards_HomeTeam_median_10,Away_Yellow_Cards_HomeTeam_median_10,Home_Red_Cards_HomeTeam_median_10,Away_Red_Cards_HomeTeam_median_10,Home_Bookings_Points_HomeTeam_median_10,Away_Bookings_Points_HomeTeam_median_10,Final_result_HomeTeam_median_10,Home_score_HomeTeam_median_10,Away_score_HomeTeam_median_10,Final_Home_goals_AwayTeam_median_10,Final_Away_goals_AwayTeam_median_10,Half_Time_Home_Goals_AwayTeam_median_10,Half_Time_Away_Goals_AwayTeam_median_10,Half_Time_Result_AwayTeam_median_10,Crowd_Attendance_AwayTeam_median_10,Home_Shots_AwayTeam_median_10,Away_Shots_AwayTeam_median_10,Home_Shots_Target_AwayTeam_median_10,Away_Shots_Target_AwayTeam_median_10,Home_Woodwork_AwayTeam_median_10,Away_Woodwork_AwayTeam_median_10,Home_Corners_AwayTeam_median_10,Away_Corners_AwayTeam_median_10,Home_Fouls_AwayTeam_median_10,Away_Fouls_AwayTeam_median_10,Home_Offsides_AwayTeam_median_10,Away_Offsides_AwayTeam_median_10,Home_Yellow_Cards_AwayTeam_median_10,Away_Yellow_Cards_AwayTeam_median_10,Home_Red_Cards_AwayTeam_median_10,Away_Red_Cards_AwayTeam_median_10,Home_Bookings_Points_AwayTeam_median_10,Away_Bookings_Points_AwayTeam_median_10,Final_result_AwayTeam_median_10,Home_score_AwayTeam_median_10,Away_score_AwayTeam_median_10,Fin

In [118]:
df_naha.loc[:,df_naha.columns.str.contains('Final_Home_goals')].columns

Index(['Final_Home_goals', 'Final_Home_goals_HomeTeam_sum_10',
       'Final_Home_goals_AwayTeam_sum_10',
       'Final_Home_goals_HomeTeam_median_10',
       'Final_Home_goals_AwayTeam_median_10',
       'Final_Home_goals_HomeTeam_average_10',
       'Final_Home_goals_AwayTeam_average_10',
       'Final_Home_goals_Home_AllGames_Average_10',
       'Final_Home_goals_Home_AllGames_sum_10',
       'Final_Home_goals_Home_AllGames_median_10',
       'Final_Home_goals_Away_AllGames_Average_10',
       'Final_Home_goals_Away_AllGames_sum_10',
       'Final_Home_goals_Away_AllGames_median_10',
       'Final_Home_goals_10Away_of_home_sum',
       'Final_Home_goals_10Away_of_home_median',
       'Final_Home_goals_10Away_of_home_mean',
       'Final_Home_goals_10Home_of_away_sum',
       'Final_Home_goals_10Home_of_away_median',
       'Final_Home_goals_10Home_of_away_mean'],
      dtype='object')

In [119]:
cols= set(df_naha.columns.to_list()) - set(df_naha.loc[:,'AwayTeam':'Away_score'].columns.to_list())

In [120]:
df_naha=df_naha.loc[:,cols]

In [121]:
df_ready= pd.merge(df, df_naha, how='left', on=['Date', 'HomeTeam'])

In [122]:
a1=set(df_ready.loc[:,'Final_Home_goals':'Half_Time_Result'].columns.to_list())

In [123]:
a2=set(df_ready.loc[:,'Home_Shots':'Away_Bookings_Points'].columns.to_list())

In [124]:
a3=set(df_ready.loc[:,'Winner':'Corners_ratio'].columns.to_list())

In [125]:
cols= set(df_ready.columns.to_list()) - a1

In [126]:
cols= cols- a2

In [127]:
cols= cols- a3

In [128]:
df_ready=df_ready.loc[:,cols]

In [129]:
L= ['Date_'+str(rolling_num)+'Home_of_away_sum', 'Date_'+str(rolling_num)+'Away_of_home_mean', 'Date_'+str(rolling_num)+'Home_of_away_median', 'Date_'+str(rolling_num)+'Away_of_home_sum', 'Date_'+str(rolling_num)+'Home_of_away_mean',
    'Date_'+str(rolling_num)+'Away_of_home_median', 'menha_date_'+str(rolling_num)+'Home_of_away_mean', 'menha_date_'+str(rolling_num)+'Away_of_home_mean', 'menha_date_'+str(rolling_num)+'Home_of_away_sum',
    'menha_date_'+str(rolling_num)+'Away_of_home_sum', 'menha_date_'+str(rolling_num)+'Away_of_home_median', 'menha_date_'+str(rolling_num)+'Home_of_away_median']


In [130]:
cols= set(df_ready.columns.to_list()) - set(L)

In [131]:
df_ready=df_ready.loc[:,cols]

In [132]:
df_ready.to_csv('df_ready_No_dummy_4.csv')

In [133]:
df_ready= pd.read_csv('df_ready_No_dummy_4.csv')

C:\Users\enabi\anaconda3\envs\untitled\lib\site-packages\IPython\core\interactiveshell.py:3072: DtypeWarning: Columns (432) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [134]:
del df_ready['Unnamed: 0']

In [135]:
#HOME
# _HomeTeam_sum_6
# _HomeTeam_median_6
# _HomeTeam_average_6
# _Home_AllGames_sum_6
# _Home_AllGames_median_6
# _Home_AllGames_Average_6
#_6Away_of_home_sum
# _6Away_of_home_median
# _6Away_of_home_mean

filt1 = df_ready.columns[df_ready.columns.str.contains(pat = '_HomeTeam_sum_'+str(rolling_num)+'|_HomeTeam_median_'+str(rolling_num)+'|_HomeTeam_average_'+str(rolling_num)+'|_Home_AllGames_sum_'+str(rolling_num)+'|_Home_AllGames_median_'+str(rolling_num)+'|_Home_AllGames_Average_'+str(rolling_num)+'|_'+str(rolling_num)+'Away_of_home_sum|_'+str(rolling_num)+'Away_of_home_median|_'+str(rolling_num)+'Away_of_home_mean')]


In [136]:
#AWAY
# _AwayTeam_sum_6
# _AwayTeam_median_6
# _AwayTeam_average_6
# _Away_AllGames_sum_6
# _Away_AllGames_median_6
# _Away_AllGames_Average_6
# _6Home_of_away_sum
# _6Home_of_away_median
# _6Home_of_away_mean

filt2 = df_ready.columns[df_ready.columns.str.contains(pat = '_AwayTeam_sum_'+str(rolling_num)+'|_AwayTeam_median_'+str(rolling_num)+'|_AwayTeam_average_'+str(rolling_num)+'|_Away_AllGames_sum_'+str(rolling_num)+'|_Away_AllGames_median_'+str(rolling_num)+'|_Away_AllGames_Average_'+str(rolling_num)+'|_'+str(rolling_num)+'Home_of_away_sum|_'+str(rolling_num)+'Home_of_away_median|_'+str(rolling_num)+'Home_of_away_mean')]


In [137]:
df_N= df_ready.copy()

In [138]:
df_N.loc[:,list(filt1)]= df_N.loc[:,list(filt1)+['HomeTeam']].fillna(df_N.groupby(['HomeTeam'])[filt1].transform('mean'))

In [139]:
df_N.loc[:,list(filt2)]= df_N.loc[:,list(filt2)+['AwayTeam']].fillna(df_N.groupby(['AwayTeam'])[filt2].transform('mean'))

In [140]:
pd.set_option("max_rows", None)

In [141]:
df_p= df_N.fillna(-1)

In [142]:
#dummies:

mylist = list(['HomeTeam','AwayTeam','Referee'])
dummies = pd.get_dummies(df_p[mylist], prefix= mylist)
df_p.drop(mylist, axis=1, inplace = True)
df_p = pd.concat([df_p,dummies], axis =1 )

In [143]:
df_p.shape

(7508, 851)

In [144]:
df_p.to_csv('df_p_10.csv')